In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


loading data

In [3]:
from pathlib import Path

data_path = Path(r"C:\Users\Niraj Mhatre\projects\Yield-Curve-Forecasting-Using-Macroeconomic-Indicators\Data\Raw1")
dgs3mo = pd.read_csv(data_path / "DGS3MO.csv")
dgs1 = pd.read_csv(data_path / "DGS1.csv")
dgs2 = pd.read_csv(data_path / "DGS2.csv")
dgs5 = pd.read_csv(data_path / "DGS5.csv")
dgs10 = pd.read_csv(data_path / "DGS10.csv")
dgs30 = pd.read_csv(data_path / "DGS30.csv")

cpiaucsl = pd.read_csv(data_path / "CPIAUCSL.csv")
cpilfesl = pd.read_csv(data_path / "CPILFESL.csv")
fedfunds = pd.read_csv(data_path / "FEDFUNDS.csv")
unrate = pd.read_csv(data_path / "UNRATE.csv")
gdpc1 = pd.read_csv(data_path / "GDPC1.csv")

indpro = pd.read_csv(data_path / "INDPRO.csv")
rsafs = pd.read_csv(data_path / "RSAFS.csv")
houst = pd.read_csv(data_path / "HOUST.csv")
icsa = pd.read_csv(data_path / "ICSA.csv")
umcsent = pd.read_csv(data_path / "UMCSENT.csv")

vixcls = pd.read_csv(data_path / "VIXCLS.csv")
dcoilwtico = pd.read_csv(data_path / "DCOILWTICO.csv")

In [4]:
#converting to date-time format
dfs = [
    dgs3mo, dgs1, dgs2, dgs5, dgs10, dgs30,
    cpiaucsl, cpilfesl, fedfunds, unrate, gdpc1,
    indpro, rsafs, houst, icsa, umcsent,
    vixcls, dcoilwtico
]

for df in dfs:
    df["observation_date"] = pd.to_datetime(df["observation_date"])

| Variable               | Frequency | Monthly Conversion                                          |
| ---------------------- | --------- | ----------------------------------------------------------- |
| Treasury Yields (DGS*) | Daily     | Last observation of the month                               |
| VIX                    | Daily     | Monthly average                                             |
| Dollar Index           | Daily     | Monthly average                                             |
| WTI Oil                | Daily     | Monthly average                                             |
| CPI                    | Monthly   | No change                                                   |
| Core CPI               | Monthly   | No change                                                   |
| Fed Funds              | Monthly   | No change                                                   |
| Unemployment           | Monthly   | No change                                                   |
| Industrial Production  | Monthly   | No change                                                   |
| Retail Sales           | Monthly   | No change                                                   |
| Housing Starts         | Monthly   | No change                                                   |
| Consumer Sentiment     | Monthly   | No change                                                   |
| GDP                    | Quarterly | Forward-fill to monthly after merge (don't interpolate yet) |

we convert data to monthly data in the given format

In [5]:
dgs3mo = dgs3mo.set_index("observation_date").resample("ME").last().reset_index()
dgs1 = dgs1.set_index("observation_date").resample("ME").last().reset_index()
dgs2 = dgs2.set_index("observation_date").resample("ME").last().reset_index()
dgs5 = dgs5.set_index("observation_date").resample("ME").last().reset_index()
dgs10 = dgs10.set_index("observation_date").resample("ME").last().reset_index()
dgs30 = dgs30.set_index("observation_date").resample("ME").last().reset_index()


In [6]:
vixcls = vixcls.set_index("observation_date").resample("ME").mean().reset_index()

dcoilwtico = dcoilwtico.set_index("observation_date").resample("ME").mean().reset_index()

In [7]:
monthly_dfs = [
    cpiaucsl,
    cpilfesl,
    fedfunds,
    unrate,
    indpro,
    rsafs,
    houst,
    umcsent
]

a few sanity checks before merging the data

In [8]:
# checking for duplicates
for name, df in {
    "dgs3mo": dgs3mo,
    "dgs1": dgs1,
    "dgs2": dgs2,
    "dgs5": dgs5,
    "dgs10": dgs10,
    "dgs30": dgs30,
    "cpiaucsl": cpiaucsl,
    "cpilfesl": cpilfesl,
    "fedfunds": fedfunds,
    "unrate": unrate,
    "gdpc1": gdpc1,
    "indpro": indpro,
    "rsafs": rsafs,
    "houst": houst,
    "umcsent": umcsent,
    "vixcls": vixcls,
    "dcoilwtico": dcoilwtico
}.items():
    print(name, df["observation_date"].duplicated().sum())

dgs3mo 0
dgs1 0
dgs2 0
dgs5 0
dgs10 0
dgs30 0
cpiaucsl 0
cpilfesl 0
fedfunds 0
unrate 0
gdpc1 0
indpro 0
rsafs 0
houst 0
umcsent 0
vixcls 0
dcoilwtico 0


In [9]:
# date ranges
for name, df in {
    "dgs3mo": dgs3mo,
    "dgs1": dgs1,
    "dgs2": dgs2,
    "dgs5": dgs5,
    "dgs10": dgs10,
    "dgs30": dgs30,
    "cpiaucsl": cpiaucsl,
    "cpilfesl": cpilfesl,
    "fedfunds": fedfunds,
    "unrate": unrate,
    "gdpc1": gdpc1,
    "indpro": indpro,
    "rsafs": rsafs,
    "houst": houst,
    "umcsent": umcsent,
    "vixcls": vixcls,
    "dcoilwtico": dcoilwtico
}.items():
    print(name, df["observation_date"].min(), df["observation_date"].max())

dgs3mo 1990-01-31 00:00:00 2026-07-31 00:00:00
dgs1 1990-01-31 00:00:00 2026-07-31 00:00:00
dgs2 1990-01-31 00:00:00 2026-07-31 00:00:00
dgs5 1990-01-31 00:00:00 2026-07-31 00:00:00
dgs10 1990-01-31 00:00:00 2026-07-31 00:00:00
dgs30 1990-01-31 00:00:00 2026-07-31 00:00:00
cpiaucsl 1990-01-01 00:00:00 2026-06-01 00:00:00
cpilfesl 1990-01-01 00:00:00 2026-06-01 00:00:00
fedfunds 1990-01-01 00:00:00 2026-06-01 00:00:00
unrate 1990-01-01 00:00:00 2026-06-01 00:00:00
gdpc1 1990-01-01 00:00:00 2026-04-01 00:00:00
indpro 1990-01-01 00:00:00 2026-06-01 00:00:00
rsafs 1992-01-01 00:00:00 2026-06-01 00:00:00
houst 1990-01-01 00:00:00 2026-06-01 00:00:00
umcsent 1990-01-01 00:00:00 2026-06-01 00:00:00
vixcls 1990-01-31 00:00:00 2026-07-31 00:00:00
dcoilwtico 1990-01-31 00:00:00 2026-07-31 00:00:00


In [10]:
# mising values
for name, df in {
    "dgs3mo": dgs3mo,
    "dgs1": dgs1,
    "dgs2": dgs2,
    "dgs5": dgs5,
    "dgs10": dgs10,
    "dgs30": dgs30,
    "cpiaucsl": cpiaucsl,
    "cpilfesl": cpilfesl,
    "fedfunds": fedfunds,
    "unrate": unrate,
    "gdpc1": gdpc1,
    "indpro": indpro,
    "rsafs": rsafs,
    "houst": houst,
    "umcsent": umcsent,
    "vixcls": vixcls,
    "dcoilwtico": dcoilwtico
}.items():
    print(name)
    print(df.isna().sum())

dgs3mo
observation_date    0
DGS3MO              0
dtype: int64
dgs1
observation_date    0
DGS1                0
dtype: int64
dgs2
observation_date    0
DGS2                0
dtype: int64
dgs5
observation_date    0
DGS5                0
dtype: int64
dgs10
observation_date    0
DGS10               0
dtype: int64
dgs30
observation_date    0
DGS30               0
dtype: int64
cpiaucsl
observation_date    0
CPIAUCSL            1
dtype: int64
cpilfesl
observation_date    0
CPILFESL            1
dtype: int64
fedfunds
observation_date    0
FEDFUNDS            0
dtype: int64
unrate
observation_date    0
UNRATE              1
dtype: int64
gdpc1
observation_date    0
GDPC1               0
dtype: int64
indpro
observation_date    0
INDPRO              0
dtype: int64
rsafs
observation_date    0
RSAFS               0
dtype: int64
houst
observation_date    0
HOUST               0
dtype: int64
umcsent
observation_date    0
UMCSENT             0
dtype: int64
vixcls
observation_date    0
VIXCLS         

In [11]:
datasets = {
    "DGS3MO": dgs3mo,
    "DGS1": dgs1,
    "DGS2": dgs2,
    "DGS5": dgs5,
    "DGS10": dgs10,
    "DGS30": dgs30,
    "CPIAUCSL": cpiaucsl,
    "CPILFESL": cpilfesl,
    "FEDFUNDS": fedfunds,
    "UNRATE": unrate,
    "GDPC1": gdpc1,
    "INDPRO": indpro,
    "RSAFS": rsafs,
    "HOUST": houst,
    "UMCSENT": umcsent,
    "VIXCLS": vixcls,
    "DCOILWTICO": dcoilwtico
}

for name, df in datasets.items():
    value_col = [c for c in df.columns if c != "observation_date"][0]
    df.rename(columns={value_col: name}, inplace=True)

In [12]:
for df in datasets.values():
    df["observation_date"] = df["observation_date"].dt.to_period("M")

In [13]:
master_df = list(datasets.values())[0]

for df in list(datasets.values())[1:]:
    master_df = master_df.merge(df, on="observation_date", how="outer")

In [14]:
master_df = master_df.sort_values("observation_date").reset_index(drop=True)

In [15]:
master_df.head(20)

,observation_date,DGS3MO,DGS1,DGS2,DGS5,DGS10,DGS30,CPIAUCSL,CPILFESL,FEDFUNDS,UNRATE,GDPC1,INDPRO,RSAFS,HOUST,UMCSENT,VIXCLS,DCOILWTICO
0,1990-01,8.00,8.08,8.28,8.35,8.43,8.46,127.5,132.1,8.23,5.4,10047.386,61.7290,NaN,1551.0,93.0,23.347273,22.863182
1,1990-02,8.04,8.12,8.43,8.44,8.51,8.54,128.0,132.7,8.24,5.3,NaN,62.2896,NaN,1437.0,89.5,23.262632,22.113000
2,1990-03,8.07,8.35,8.64,8.65,8.65,8.63,128.6,133.5,8.28,5.2,NaN,62.5999,NaN,1289.0,91.3,20.062273,20.387727
3,1990-04,8.07,8.58,8.96,9.04,9.04,9.00,128.9,134.0,8.26,5.4,10083.855,62.4359,NaN,1248.0,93.9,21.403500,18.425500
4,1990-05,8.01,8.22,8.50,8.56,8.60,8.58,129.1,134.4,8.18,5.4,NaN,62.6258,NaN,1212.0,90.6,18.097727,18.199545
5,1990-06,8.00,8.05,8.24,8.35,8.43,8.41,129.9,135.1,8.29,5.2,NaN,62.8382,NaN,1177.0,88.3,16.822381,16.695238
6,1990-07,7.74,7.72,7.91,8.13,8.36,8.42,130.5,135.8,8.15,5.5,10090.569,62.7284,NaN,1171.0,88.2,18.392857,18.454091
7,1990-08,7.63,7.76,8.07,8.50,8.86,8.99,131.6,136.6,8.13,5.7,NaN,62.9475,NaN,1115.0,76.4,28.175217,27.307391
8,1990-09,7.37,7.69,8.02,8.47,8.82,8.96,132.5,137.1,8.20,5.9,NaN,62.9523,NaN,1110.0,72.8,29.107368,33.507500
9,1990-10,7.34,7.43,7.77,8.24,8.65,8.78,133.4,137.6,8.11,5.9,9998.704,62.5845,NaN,1014.0,63.9,29.625652,36.039565


In [16]:
master_df["GDPC1"] = master_df["GDPC1"].ffill()

In [17]:
master_df[["observation_date", "GDPC1"]].head(20)

,observation_date,GDPC1
0,1990-01,10047.386
1,1990-02,10047.386
2,1990-03,10047.386
3,1990-04,10083.855
4,1990-05,10083.855
5,1990-06,10083.855
6,1990-07,10090.569
7,1990-08,10090.569
8,1990-09,10090.569
9,1990-10,9998.704


In [18]:
master_df["GDPC1"].isna().sum()

np.int64(0)

In [19]:
master_df.head(30)

,observation_date,DGS3MO,DGS1,DGS2,DGS5,DGS10,DGS30,CPIAUCSL,CPILFESL,FEDFUNDS,UNRATE,GDPC1,INDPRO,RSAFS,HOUST,UMCSENT,VIXCLS,DCOILWTICO
0,1990-01,8.00,8.08,8.28,8.35,8.43,8.46,127.5,132.1,8.23,5.4,10047.386,61.7290,NaN,1551.0,93.0,23.347273,22.863182
1,1990-02,8.04,8.12,8.43,8.44,8.51,8.54,128.0,132.7,8.24,5.3,10047.386,62.2896,NaN,1437.0,89.5,23.262632,22.113000
2,1990-03,8.07,8.35,8.64,8.65,8.65,8.63,128.6,133.5,8.28,5.2,10047.386,62.5999,NaN,1289.0,91.3,20.062273,20.387727
3,1990-04,8.07,8.58,8.96,9.04,9.04,9.00,128.9,134.0,8.26,5.4,10083.855,62.4359,NaN,1248.0,93.9,21.403500,18.425500
4,1990-05,8.01,8.22,8.50,8.56,8.60,8.58,129.1,134.4,8.18,5.4,10083.855,62.6258,NaN,1212.0,90.6,18.097727,18.199545
5,1990-06,8.00,8.05,8.24,8.35,8.43,8.41,129.9,135.1,8.29,5.2,10083.855,62.8382,NaN,1177.0,88.3,16.822381,16.695238
6,1990-07,7.74,7.72,7.91,8.13,8.36,8.42,130.5,135.8,8.15,5.5,10090.569,62.7284,NaN,1171.0,88.2,18.392857,18.454091
7,1990-08,7.63,7.76,8.07,8.50,8.86,8.99,131.6,136.6,8.13,5.7,10090.569,62.9475,NaN,1115.0,76.4,28.175217,27.307391
8,1990-09,7.37,7.69,8.02,8.47,8.82,8.96,132.5,137.1,8.20,5.9,10090.569,62.9523,NaN,1110.0,72.8,29.107368,33.507500
9,1990-10,7.34,7.43,7.77,8.24,8.65,8.78,133.4,137.6,8.11,5.9,9998.704,62.5845,NaN,1014.0,63.9,29.625652,36.039565


In [22]:
[bf1992] = np.where(master_df["RSAFS"].isna())

KeyError: 'RSAFS'

In [20]:
from pathlib import Path

processed_path = Path(r"C:\Users\Niraj Mhatre\projects\Yield-Curve-Forecasting-Using-Macroeconomic-Indicators\Data\Processed")

processed_path.mkdir(parents=True, exist_ok=True)

master_df.to_csv(processed_path / "master_dataset_outer.csv", index=False)